# Notebook 03 — Evaluation: BiEncoder vs BM25

**Goal:** Measure retrieval quality on the 10 hand-crafted test queries and compare
the trained BiEncoder against the BM25 baseline.

## Metrics

| Metric | What it measures |
|---|---|
| **MRR@10** | Mean Reciprocal Rank — average of 1/rank of the correct result |
| **R@1** | Was the top-1 result correct? (= precision / accuracy) |
| **R@5** | Was the correct result anywhere in the top 5? |
| **R@10** | Was the correct result anywhere in the top 10? |

## Expected results
- BM25 is a strong baseline for technical documents — don't be surprised if it beats
  an under-trained BiEncoder on exact-match queries.
- BiEncoder should outperform BM25 on **semantic / paraphrase** queries where the
  exact keywords don't appear in the relevant chunk.

---
> Can run on CPU. GPU not needed for evaluation.

## Setup

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Neural_Search_Engine'
    sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)
    !pip install -q transformers tokenizers safetensors rank-bm25
else:
    PROJECT_ROOT = os.path.abspath('..')
    sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)

import torch
import json
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer

from src.model import BiEncoder
from src.vector_store import VectorStore
from src.evaluate import evaluate_biencoder, evaluate_bm25, comparison_table, qualitative_comparison

print('Setup complete.')

---
## 1. Load Data and Models

In [ ]:
# Load corpus
with open('data/processed/jurafsky_chunks.json', encoding='utf-8') as f:
    chunks = json.load(f)

# Load evaluation set (10 hand-crafted queries)
eval_df = pd.read_csv('data/evaluation_set.csv')

print(f'Corpus chunks : {len(chunks)}')
print(f'Test queries  : {len(eval_df)}')
print()
print('Test queries:')
for i, row in eval_df.iterrows():
    print(f"  [{i+1}] {row['query'][:90]}")
    print(f"       expected: {row['expected_chunk_id']}")

In [ ]:
# ── Tokenizer ────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

# ── Pre-trained (no fine-tuning) BiEncoder ───────────────────────────────────
model_pretrained = BiEncoder()
model_pretrained.eval()

store_pretrained = VectorStore(model_pretrained, tokenizer, batch_size=64)
store_pretrained.build(chunks)

# ── Fine-tuned BiEncoder ─────────────────────────────────────────────────────
model_finetuned = BiEncoder()
model_finetuned.load_state_dict(
    torch.load('checkpoints/best.pt', map_location='cpu')
)
model_finetuned.eval()

store_finetuned = VectorStore(model_finetuned, tokenizer, batch_size=64)
store_finetuned.build(chunks)

---
## 2. Compute Metrics for All Three Systems

In [ ]:
print('Evaluating BM25...')
bm25_metrics = evaluate_bm25(chunks, eval_df)

print('Evaluating pre-trained BiEncoder (no fine-tuning)...')
pretrained_metrics = evaluate_biencoder(store_pretrained, eval_df)

print('Evaluating fine-tuned BiEncoder...')
finetuned_metrics = evaluate_biencoder(store_finetuned, eval_df)

print('Done.')

In [ ]:
# Build a three-way comparison table
metrics_keys = list(bm25_metrics.keys())
table = pd.DataFrame({
    'Metric':              metrics_keys,
    'BM25':                [bm25_metrics[k]        for k in metrics_keys],
    'BiEnc (pretrained)':  [pretrained_metrics[k]  for k in metrics_keys],
    'BiEnc (fine-tuned)':  [finetuned_metrics[k]   for k in metrics_keys],
})
table['Δ (finetuned − BM25)'] = (table['BiEnc (fine-tuned)'] - table['BM25']).round(4)

print('\n=== Retrieval Evaluation Results ===')
print(table.to_string(index=False))

---
## 3. Visualise Results

In [ ]:
import numpy as np

x = np.arange(len(metrics_keys))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width, table['BM25'],               width, label='BM25',               color='#aec7e8')
ax.bar(x,         table['BiEnc (pretrained)'],  width, label='BiEnc (pretrained)', color='#ffbb78')
ax.bar(x + width, table['BiEnc (fine-tuned)'],  width, label='BiEnc (fine-tuned)', color='#98df8a')

ax.set_xticks(x)
ax.set_xticklabels(metrics_keys)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score (higher is better)')
ax.set_title('Retrieval Evaluation: BM25 vs BiEncoder')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('checkpoints/evaluation_results.png', dpi=150)
plt.show()

---
## 4. Per-Query Analysis

In [ ]:
from src.evaluate import reciprocal_rank

print(f'{'Query':<60} {'BM25 rank':>9} {'BiEnc rank':>10}')
print('-' * 82)

for _, row in eval_df.iterrows():
    query = row['query']
    correct_id = row['expected_chunk_id']

    # BM25
    from src.evaluate import _tokenize_bm25
    from rank_bm25 import BM25Okapi
    corpus = [c['content'] for c in chunks]
    chunk_ids = [c['id'] for c in chunks]
    toks = [_tokenize_bm25(d) for d in corpus]
    bm25 = BM25Okapi(toks)
    scores = bm25.get_scores(_tokenize_bm25(query))
    bm25_ranked = [chunk_ids[i] for i in sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:10]]
    bm25_rank = next((i+1 for i, cid in enumerate(bm25_ranked) if cid == correct_id), '>10')

    # BiEncoder
    bi_results = store_finetuned.search(query, top_k=10)
    bi_ranked = [r['id'] for r in bi_results]
    bi_rank = next((i+1 for i, cid in enumerate(bi_ranked) if cid == correct_id), '>10')

    winner = '← BiEnc wins' if (isinstance(bi_rank, int) and (not isinstance(bm25_rank, int) or bi_rank < bm25_rank)) else \
             ('← BM25 wins' if (isinstance(bm25_rank, int) and (not isinstance(bi_rank, int) or bm25_rank < bi_rank)) else '')

    print(f"{query[:58]:<60} {str(bm25_rank):>9} {str(bi_rank):>10}  {winner}")

---
## 5. Qualitative Comparison — Side-by-Side Results

In [ ]:
# Pick 2 interesting queries and show what each system returns
queries_to_show = eval_df['query'].tolist()[:2]

for query in queries_to_show:
    print('=' * 80)
    print(f'QUERY: {query}')
    print()

    bi_results = store_finetuned.search(query, top_k=3)

    from rank_bm25 import BM25Okapi
    corpus = [c['content'] for c in chunks]
    chunk_ids = [c['id'] for c in chunks]
    bm25 = BM25Okapi([_tokenize_bm25(d) for d in corpus])
    bm25_scores = bm25.get_scores(_tokenize_bm25(query))
    top3 = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:3]

    print(f'{"BM25 results":<50} | {"BiEncoder results"}')
    print('-' * 100)
    for rank in range(3):
        bm_id = chunk_ids[top3[rank]]
        bm_text = corpus[top3[rank]][:80]
        bi_id = bi_results[rank]['id']
        bi_text = bi_results[rank]['content'][:80]
        print(f'[{rank+1}] {bm_id}: {bm_text}...')
        print(f'    BiEnc [{rank+1}] {bi_id}: {bi_text}...')
        print()
    print()

---
Evaluation complete. See `checkpoints/evaluation_results.png` for the bar chart.  
Proceed to **04_demo.ipynb** for the interactive search demo.